In [13]:
# Cell 1: Setup
from openai import OpenAI
import time

URL_AWQ = "https://qwen3--vl--32b--instruct-awq-runai-shared-models.deepthought.doit.wisc.edu/v1"
URL_8BIT = "https://qwen3--vl--32b--instruct-8bit-runai-shared-models.deepthought.doit.wisc.edu/v1"

PROMPT = (
    "Write a detailed explanation of how transformer attention mechanisms work, "
    "including multi-head attention, scaled dot-product attention, and the role "
    "of query, key, and value matrices. Be thorough."
)
MAX_TOKENS = 256
RUNS = 10

In [14]:
# Cell 2: Benchmark function
def bench(url, runs=RUNS, max_tokens=MAX_TOKENS, prompt=PROMPT):
    client = OpenAI(base_url=url, api_key="not-used")
    model = client.models.list().data[0].id
    print(f"Model: {model}\nURL:   {url}\n")

    results = []
    for i in range(runs):
        t0 = time.perf_counter()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0.0,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.perf_counter() - t0
        toks = resp.usage.completion_tokens
        tok_s = toks / elapsed
        print(f"  run {i+1}/{runs}: {toks} tokens in {elapsed:.2f}s → {tok_s:.1f} tok/s")
        results.append({"toks": toks, "elapsed": elapsed, "tok_s": tok_s})

    avg = sum(r["tok_s"] for r in results) / len(results)
    print(f"\n  AVG: {avg:.1f} tok/s\n")
    return {"model": model, "avg_tok_s": avg, "runs": results}

In [15]:
# Cell 3: Run AWQ (4-bit)
awq = bench(URL_AWQ)


Model: QuantTrio/Qwen3-VL-32B-Instruct-AWQ
URL:   https://qwen3--vl--32b--instruct-awq-runai-shared-models.deepthought.doit.wisc.edu/v1

  run 1/10: 256 tokens in 3.94s → 64.9 tok/s
  run 2/10: 256 tokens in 3.94s → 65.0 tok/s
  run 3/10: 256 tokens in 3.94s → 65.0 tok/s
  run 4/10: 256 tokens in 3.93s → 65.1 tok/s
  run 5/10: 256 tokens in 3.94s → 65.0 tok/s
  run 6/10: 256 tokens in 3.93s → 65.1 tok/s
  run 7/10: 256 tokens in 3.93s → 65.1 tok/s
  run 8/10: 256 tokens in 3.94s → 65.1 tok/s
  run 9/10: 256 tokens in 3.94s → 65.0 tok/s
  run 10/10: 256 tokens in 3.93s → 65.1 tok/s

  AVG: 65.0 tok/s



In [16]:
# Cell 5: Compare
print(f"{'Model':<55} {'Avg tok/s':>10}")
print(f"{'-'*55} {'-'*10}")
print(f"{awq['model']:<55} {awq['avg_tok_s']:>10.1f}")


Model                                                    Avg tok/s
------------------------------------------------------- ----------
QuantTrio/Qwen3-VL-32B-Instruct-AWQ                           65.0


In [18]:
eight = bench(URL_8BIT)


Model: Qwen/Qwen3-VL-32B-Instruct
URL:   https://qwen3--vl--32b--instruct-8bit-runai-shared-models.deepthought.doit.wisc.edu/v1

  run 1/10: 256 tokens in 6.70s → 38.2 tok/s
  run 2/10: 256 tokens in 6.68s → 38.3 tok/s
  run 3/10: 256 tokens in 6.67s → 38.4 tok/s
  run 4/10: 256 tokens in 6.68s → 38.3 tok/s
  run 5/10: 256 tokens in 6.68s → 38.3 tok/s
  run 6/10: 256 tokens in 6.68s → 38.3 tok/s
  run 7/10: 256 tokens in 6.68s → 38.3 tok/s
  run 8/10: 256 tokens in 6.68s → 38.3 tok/s
  run 9/10: 256 tokens in 6.68s → 38.4 tok/s
  run 10/10: 256 tokens in 6.67s → 38.4 tok/s

  AVG: 38.3 tok/s



In [19]:
print(f"{eight['model']:<55} {eight['avg_tok_s']:>10.1f}")

ratio = awq["avg_tok_s"] / eight["avg_tok_s"]
if ratio > 1:
    print(f"\nAWQ 4-bit is {ratio:.2f}x faster")
else:
    print(f"\n8-bit is {1/ratio:.2f}x faster")

Qwen/Qwen3-VL-32B-Instruct                                    38.3

AWQ 4-bit is 1.70x faster
